
# Converting Bronze Tables to Silver Tables

Each Bronze table, Is filtered through the following:

- Inspect schema and data quality.
- Remove duplicate records.
- Handle missing or invalid values.
- Standardize column names, formats, and data types.
- Apply business rules and required transformations.
- Load cleaned data into the Silver layer.
- Validate the results to ensure data quality and consistency.

## Retrieve Bronze Tables

In [0]:
%sql 
USE CATALOG azure_databricks_jarvis;
USE SCHEMA bronze;

In [0]:
from pyspark.sql.functions import col, when, upper, isnan, when, count
from pyspark.sql.types import DateType, StringType, DoubleType, BooleanType

In [0]:
fraud = spark.read.table("fraud_table")
users = spark.read.table("users_data")
transactions = spark.read.table("transactions_data")
cards = spark.read.table("cards_data")
mcc_codes = spark.read.table("mccodes_table")

fraud.dtypes
users.dtypes
transactions.dtypes
cards.dtypes
mcc_codes.dtypes

[('Code', 'int'), ('Item', 'string')]

In [0]:
fraud.dropDuplicates()
users.dropDuplicates()
transactions.dropDuplicates()
cards.dropDuplicates()
mcc_codes.dropDuplicates()

DataFrame[Code: int, Item: string]

In [0]:
%sql
ALTER TABLE bronze.fraud_table
ALTER COLUMN TransactionID SET NOT NULL;

ALTER TABLE bronze.fraud_table
ALTER COLUMN Value SET NOT NULL;


## Convert columns to the appropriate data types.

In [0]:
## Fraud
fraud_update = (
    fraud \
    .withColumnRenamed("TransactionID", "transaction_id") \
    .withColumnRenamed("Value", "value") \
    .withColumn("transaction_id", col("transaction_id").cast("int")) \
    .withColumn(
        "value",
        when(upper(col("value")) == "YES", True)
        .when(upper(col("value")) == "NO", False)
        .otherwise(None)
    )
)


In [0]:
## MCC Codes
mcc_codes_updates = mcc_codes \
    .withColumnRenamed("Code", "code") \
    .withColumnRenamed("Item", "item") \
    .withColumn("code", col("code").cast("Integer")) 

In [0]:
## users Table
users.printSchema()

root
 |-- id: short (nullable = true)
 |-- current_age: short (nullable = true)
 |-- retirement_age: short (nullable = true)
 |-- birth_year: short (nullable = true)
 |-- birth_month: short (nullable = true)
 |-- gender: string (nullable = true)
 |-- address: string (nullable = true)
 |-- latitude: double (nullable = true)
 |-- longitude: double (nullable = true)
 |-- per_capita_income: decimal(19,4) (nullable = true)
 |-- yearly_income: decimal(19,4) (nullable = true)
 |-- total_debt: decimal(19,4) (nullable = true)
 |-- credit_score: short (nullable = true)
 |-- num_credit_cards: short (nullable = true)



In [0]:
## transactions Table
transactions.printSchema()

root
 |-- id: integer (nullable = true)
 |-- date: timestamp (nullable = true)
 |-- client_id: short (nullable = true)
 |-- card_id: short (nullable = true)
 |-- amount: decimal(19,4) (nullable = true)
 |-- use_chip: string (nullable = true)
 |-- merchant_id: integer (nullable = true)
 |-- merchant_city: string (nullable = true)
 |-- merchant_state: string (nullable = true)
 |-- zip: double (nullable = true)
 |-- mcc: short (nullable = true)
 |-- errors: string (nullable = true)



In [0]:
## cards Table
cards.printSchema()

root
 |-- id: short (nullable = true)
 |-- client_id: short (nullable = true)
 |-- card_brand: string (nullable = true)
 |-- card_type: string (nullable = true)
 |-- card_number: long (nullable = true)
 |-- expires: string (nullable = true)
 |-- cvv: short (nullable = true)
 |-- has_chip: boolean (nullable = true)
 |-- num_cards_issued: short (nullable = true)
 |-- credit_limit: decimal(19,4) (nullable = true)
 |-- acct_open_date: string (nullable = true)
 |-- year_pin_last_changed: short (nullable = true)
 |-- card_on_dark_web: string (nullable = true)



## save tables


In [0]:
cards.write.mode("overwrite").saveAsTable("silver.cards_table")
transactions.write.mode("overwrite").saveAsTable("silver.transactions_table")
users.write.mode("overwrite").saveAsTable("silver.users_table")
mcc_codes_updates.write.mode("overwrite").saveAsTable("silver.mcc_codes_table")
fraud_update.write.mode("overwrite").saveAsTable("silver.fraud_table")